In [37]:
import pandas as pd
import os
import glob
from dateutil import parser

# Parameters
ENTRY_TIME = "09:30:00+05:30"
EXIT_TIME = "15:15:00+05:30"
VOL_LOOKBACK_DAYS = 3








In [38]:
def read_stock_data(file):
    df = pd.read_csv(file, parse_dates=["date"])
    df["date"] = pd.to_datetime(df["date"])
    df.set_index("date", inplace=True)
    df = df.sort_index()
    return df

def calculate_volatility(df):
    # Use 5-min returns
    returns = df['close'].pct_change()
    # Daily resample: group by date and calculate std of intraday returns
    daily_vol = returns.groupby(df.index.date).std()
    return daily_vol.rolling(window=VOL_LOOKBACK_DAYS).mean()


def predict_direction(df_day):
    # Basic rule: if price is above 5-period moving avg at 9:30 → buy
    entry_price = df_day.between_time("09:30", "09:30")["open"].iloc[0]
    ma5 = df_day["open"].rolling(5).mean().iloc[-1]
    if entry_price > ma5:
        return "buy"
    else:
        return "sell"

def simulate_trade(df_day, direction):
    entry_price = df_day.at_time(ENTRY_TIME)["open"].iloc[0]
    exit_price = df_day.at_time(EXIT_TIME)["open"].iloc[0]
    if direction == "buy":
        return (exit_price - entry_price) / entry_price
    else:
        return (entry_price - exit_price) / entry_price

def process_day(date, all_stock_data):
    print(f"Processing date: {date}", flush=True)
    volatilities = {}

    for name, df in all_stock_data.items():
        try:
            df_filtered = df[df.index.date <= date]
            if len(df_filtered) < VOL_LOOKBACK_DAYS * 78:  # 78 = 5-min bars/day
                continue
            vol_series = calculate_volatility(df_filtered)
            if not vol_series.empty:
                vol = vol_series.iloc[-1]
                volatilities[name] = vol
        except Exception as e:
            print(f"Error processing {name} on {date}: {e}", flush=True)
            continue

    if not volatilities:
        print(f"No volatility data for {date}", flush=True)
        return None

    most_volatile_stock = max(volatilities, key=volatilities.get)
    print(f"Most volatile: {most_volatile_stock}", flush=True)
    df = all_stock_data[most_volatile_stock]

    try:
        df_day = df[df.index.date == pd.to_datetime(date).date()]
        if df_day.empty:
            return None

        direction = predict_direction(df_day)
        ret = simulate_trade(df_day, direction)

        return {
            "date": date,
            "stock": most_volatile_stock,
            "direction": direction,
            "return": ret
        }
    except Exception as e:
        print(f"Simulation failed on {date} for {most_volatile_stock}: {e}", flush=True)
        return None


In [39]:

files = glob.glob("/content/*_with_indicators_.csv")
all_stock_data = {os.path.basename(f).split(".")[0]: read_stock_data(f) for f in files}
all_dates = sorted(set.union(*[set(df.index.date) for df in all_stock_data.values()]))


results = []
capital = 1000
leverage = 5


for date in all_dates[:365]:
    res = process_day(date, all_stock_data)
    if res:
        daily_return = res["return"] * leverage
        capital *= (1 + daily_return)
        res["capital"] = capital  # Track capital over time
        res["daily_leveraged_return"] = daily_return
        results.append(res)



results_df = pd.DataFrame(results)
results_df.to_csv("strategy_results.csv", index=False)
print("Backtest complete. Saved to strategy_results.csv")



Processing date: 2015-02-02
No volatility data for 2015-02-02
Processing date: 2015-02-03
No volatility data for 2015-02-03
Processing date: 2015-02-04
No volatility data for 2015-02-04
Processing date: 2015-02-05
Most volatile: SHREECEM_with_indicators_
Processing date: 2015-02-06
Most volatile: SHREECEM_with_indicators_
Processing date: 2015-02-09
Most volatile: SHREECEM_with_indicators_
Processing date: 2015-02-10
Most volatile: VEDL_with_indicators_
Processing date: 2015-02-11
Most volatile: JINDALSTEL_with_indicators_
Processing date: 2015-02-12
Most volatile: JINDALSTEL_with_indicators_
Processing date: 2015-02-13
Most volatile: BOSCHLTD_with_indicators_
Processing date: 2015-02-16
Most volatile: BOSCHLTD_with_indicators_
Processing date: 2015-02-18
Most volatile: BOSCHLTD_with_indicators_
Processing date: 2015-02-19
Most volatile: JINDALSTEL_with_indicators_
Processing date: 2015-02-20
Most volatile: JINDALSTEL_with_indicators_
Processing date: 2015-02-23
Most volatile: JINDALST